In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="152VGjw0sNhzSy07yXhUd7aH6CMUQ4S4W", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/03_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Check
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_01_setup_check.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Embeddings and Vector Search

*Part 3 of the Vizuara series on RAG Systems*

**Estimated time: 50 minutes**

Every search engine you have ever used matches keywords. Type "automobile maintenance tips" into a keyword search engine, and it will not return a single document that says "car repair advice" — even though they mean the same thing. This notebook is about fixing that fundamental limitation.

We are going to build a system that understands *meaning*, not just words. By the end, you will have embedded real sentences into a semantic vector space, built three different FAISS index types (Flat, IVF, and HNSW), benchmarked them head-to-head, and created a working semantic search engine that finds relevant documents even when the words do not match.

In [ ]:
#@title 🎧 Listen: Ai Assistant
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_02_ai_assistant.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/rag-systems/practice/3/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_03_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Let us start with a concrete example that reveals why keyword search is broken for modern applications.

In [ ]:
#@title 🎧 Listen: Install Libs
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_04_install_libs.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Setup — run this cell first
!pip install -q sentence-transformers faiss-cpu numpy matplotlib pandas scikit-learn

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import time
import textwrap
from typing import List, Tuple, Dict

# Reproducibility
np.random.seed(42)

# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Setup complete!")

In [ ]:
#@title 🎧 Listen: Automobile Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_05_automobile_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### The "Automobile" vs "Car" Problem

Imagine you are building a search engine for a car repair manual. A customer types "automobile maintenance schedule." Your database contains hundreds of documents — but they all use the word "car," not "automobile."

In [ ]:
#@title 🎧 Listen: Keyword Search Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_06_keyword_search_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# The keyword search failure
query = "automobile maintenance schedule"
documents = [
    "Car maintenance should be done every 5,000 miles.",
    "Regular vehicle servicing prevents costly repairs.",
    "The automobile industry is worth trillions of dollars.",
    "Change your car's oil filter during routine maintenance.",
    "Proper tire rotation extends the life of your car.",
]

# Naive keyword search: check if query words appear in document
query_words = set(query.lower().split())
print(f"Query: \"{query}\"")
print(f"Query words: {query_words}")
print(f"\n--- Keyword Search Results ---")

for i, doc in enumerate(documents):
    doc_words = set(doc.lower().split())
    shared = query_words & doc_words
    match = len(shared) > 0
    print(f"  {'[MATCH]' if match else '[MISS] '} \"{doc}\"")
    if shared:
        print(f"           Shared words: {shared}")

In [ ]:
#@title 🎧 Listen: Keyword Search Result
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_07_keyword_search_result.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Look at what happened. The keyword search found document 3 ("The automobile industry is worth trillions") because it shares the word "automobile" — but that document is completely irrelevant to maintenance. Meanwhile, it missed documents 1, 4, and 5 which are exactly what the customer wants but use "car" instead of "automobile."

This is the fundamental problem that embeddings solve. Instead of matching words, we match *meanings*.

In [ ]:
#@title 🎧 Listen: Building Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_building_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

Before we write any code for embeddings, let us build a mental model.

### Words as Coordinates in Semantic Space

Imagine a vast coordinate system — not with 2 or 3 dimensions, but with hundreds. Each word, sentence, or paragraph occupies a specific point in this space. The key property: **things with similar meanings are placed near each other.**


In [ ]:
#@title 🎧 Listen: Intuition 2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_08_intuition_2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Think of it like a library, but instead of organizing books alphabetically, you organize them by *topic and meaning*:
- "car," "automobile," "vehicle," and "truck" would all be clustered together in one region
- "dog," "puppy," "canine," and "pet" would be clustered in another region
- "quantum," "entanglement," "superposition," and "qubit" would be in yet another corner

The distance between two points tells you how related their meanings are. "Car" and "automobile" are almost on top of each other. "Car" and "quantum" are very far apart.

### What Do the Dimensions Represent?

In a 384-dimensional embedding space, each dimension captures some aspect of meaning. No single dimension maps neatly to a human concept, but collectively they encode:
- **Topic:** Is this about science? cooking? sports?
- **Formality:** Is this academic writing or casual conversation?
- **Sentiment:** Is this positive or negative?
- **Specificity:** Is this a broad statement or a precise detail?

### Think About This

Before we dive into the mathematics, consider:

- If "king" minus "man" plus "woman" equals "queen" in embedding space, what does that tell us about what the dimensions encode?
- Why might 384 dimensions work better than, say, 10? What information would be lost with fewer dimensions?
- What dimension might capture the "formality" of language — distinguishing "gonna" from "going to" and "shall proceed to"?

Take a moment to think about these. The mathematics will make much more sense with this spatial intuition in mind.

In [ ]:
#@title 🎧 Listen: Mathematics
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_mathematics.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

### Cosine Similarity — The Heart of Semantic Search

We need a way to measure how "close" two vectors are in embedding space. The most common measure is **cosine similarity**. Given two vectors $\mathbf{a}$ and $\mathbf{b}$:


In [ ]:
#@title 🎧 Listen: Cosine Similarity
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_09_cosine_similarity.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{||\mathbf{a}|| \; ||\mathbf{b}||}$$

Let us break this down:

1. **Dot product** ($\mathbf{a} \cdot \mathbf{b}$): Multiply corresponding elements and sum. This measures how much the vectors "agree" in each dimension.
2. **Magnitudes** ($||\mathbf{a}||$, $||\mathbf{b}||$): The length of each vector. We divide by these to normalize — so we measure *direction*, not *magnitude*.
3. **Result**: A number between -1 and 1.

The geometric interpretation: cosine similarity measures the cosine of the angle $\theta$ between two vectors.
- **1.0** means $\theta = 0°$ — vectors point the same direction (identical meaning)
- **0.0** means $\theta = 90°$ — vectors are perpendicular (unrelated)
- **-1.0** means $\theta = 180°$ — vectors point opposite directions (opposite meaning)

### Why Not Euclidean Distance?

You might wonder: why not just compute the straight-line distance between two points? The problem is that Euclidean distance is sensitive to vector magnitude. A long document might have a larger embedding magnitude than a short one, even if they discuss the same topic. Cosine similarity only cares about *direction*, making it robust to differences in text length.

In [ ]:
#@title 🎧 Listen: Worked Example
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_10_worked_example.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Worked Example: Computing Similarity by Hand

Let us plug in some simple numbers to see how this works. Take two 3D vectors:

$\mathbf{a} = [1, 2, 3]$ and $\mathbf{b} = [2, 4, 6]$

**Step 1 — Dot product:**

$$\mathbf{a} \cdot \mathbf{b} = 1 \times 2 + 2 \times 4 + 3 \times 6 = 2 + 8 + 18 = 28$$

**Step 2 — Magnitudes:**

$$||\mathbf{a}|| = \sqrt{1^2 + 2^2 + 3^2} = \sqrt{14} \approx 3.742$$

$$||\mathbf{b}|| = \sqrt{2^2 + 4^2 + 6^2} = \sqrt{56} \approx 7.483$$

**Step 3 — Divide:**

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{28}{3.742 \times 7.483} = \frac{28}{28.0} = 1.0$$

A similarity of **1.0** — perfectly aligned. Notice that $\mathbf{b} = 2 \times \mathbf{a}$. Cosine similarity does not care about magnitude, only direction. This is exactly what we want.

Now try a different pair: $\mathbf{a} = [1, 2, 3]$ and $\mathbf{c} = [3, -1, 0]$

**Dot product:** $1 \times 3 + 2 \times (-1) + 3 \times 0 = 3 - 2 + 0 = 1$

**Magnitudes:** $||\mathbf{a}|| = \sqrt{14} \approx 3.742$, $||\mathbf{c}|| = \sqrt{9 + 1 + 0} = \sqrt{10} \approx 3.162$

**Similarity:** $\frac{1}{3.742 \times 3.162} = \frac{1}{11.83} \approx 0.085$

A similarity of **0.085** — barely related. These vectors point in almost perpendicular directions.

In [ ]:
#@title 🎧 Listen: Verify Example Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_11_verify_example_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Verify the worked examples with code
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 4.0, 6.0])
c = np.array([3.0, -1.0, 0.0])

def cosine_sim_manual(v1, v2):
    """Compute cosine similarity step by step."""
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    return dot / (norm1 * norm2)

sim_ab = cosine_sim_manual(a, b)
sim_ac = cosine_sim_manual(a, c)

print("Worked Example Verification")
print("=" * 40)
print(f"a = {a}")
print(f"b = {b}")
print(f"c = {c}")
print(f"\nsim(a, b) = {sim_ab:.4f}  (perfectly aligned)")
print(f"sim(a, c) = {sim_ac:.4f}  (barely related)")

In [ ]:
#@title 🎧 Listen: Visualization Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_12_visualization_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization: How Similar Sentences Cluster in 3D

Let us create a visualization that shows how cosine similarity works geometrically. We will place several vectors in 3D space and color them by similarity to a reference vector.

In [ ]:
#@title 🎧 Listen: Visualization Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_13_visualization_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Create several 3D vectors representing different "meanings"
vectors = {
    'query':     np.array([1.0, 2.0, 3.0]),
    'similar_1': np.array([1.1, 2.2, 2.8]),   # Close to query
    'similar_2': np.array([0.9, 1.8, 3.2]),   # Close to query
    'related':   np.array([2.0, 1.0, 2.5]),   # Somewhat related
    'unrelated': np.array([3.0, -1.0, 0.5]),  # Very different
    'opposite':  np.array([-1.0, -2.0, -3.0]), # Opposite direction
}

# Compute similarities to the query vector
query_vec = vectors['query']
similarities = {}
for name, vec in vectors.items():
    similarities[name] = cosine_sim_manual(query_vec, vec)

# Plot
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(111, projection='3d')

# Color map: similarity → color
colors_map = {
    'query': '#E91E63',
    'similar_1': '#4CAF50',
    'similar_2': '#4CAF50',
    'related': '#FF9800',
    'unrelated': '#2196F3',
    'opposite': '#9C27B0',
}

for name, vec in vectors.items():
    sim = similarities[name]
    color = colors_map[name]

    # Draw vector as arrow from origin
    ax.quiver(0, 0, 0, vec[0], vec[1], vec[2],
              color=color, arrow_length_ratio=0.1, linewidth=2.5, alpha=0.8)

    # Label with name and similarity
    label = f"{name}\nsim={sim:.3f}"
    ax.text(vec[0]*1.1, vec[1]*1.1, vec[2]*1.1, label,
            fontsize=9, fontweight='bold', color=color)

ax.set_xlabel('Dimension 1', fontsize=11)
ax.set_ylabel('Dimension 2', fontsize=11)
ax.set_zlabel('Dimension 3', fontsize=11)
ax.set_title('Cosine Similarity in 3D: Direction Matters, Not Magnitude',
             fontsize=14, fontweight='bold', pad=20)

# Add legend
legend_elements = [
    mpatches.Patch(color='#E91E63', label='Query vector'),
    mpatches.Patch(color='#4CAF50', label='Similar (sim > 0.95)'),
    mpatches.Patch(color='#FF9800', label='Related (sim ~ 0.80)'),
    mpatches.Patch(color='#2196F3', label='Unrelated (sim ~ 0.08)'),
    mpatches.Patch(color='#9C27B0', label='Opposite (sim = -1.0)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

plt.tight_layout()
plt.show()

# Print summary table
print("\nSimilarity Summary:")
print("-" * 45)
for name, sim in sorted(similarities.items(), key=lambda x: x[1], reverse=True):
    bar = '#' * int(max(0, sim) * 30)
    print(f"  {name:12s}  sim = {sim:+.4f}  {bar}")

In [ ]:
#@title 🎧 Listen: Lets Build It
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_lets_build_it.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let Us Build It

Now we move from theory to practice. We will load a real embedding model, embed actual sentences, visualize what the model has learned, and then build three different vector search indices.


In [ ]:
#@title 🎧 Listen: Loading Model
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_14_loading_model.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.1 Loading the Embedding Model

We use `all-MiniLM-L6-v2` from the sentence-transformers library. This model maps any text into a 384-dimensional vector. It is small enough to run on CPU in seconds, but powerful enough for production-quality retrieval.

In [ ]:
#@title 🎧 Listen: Load Model Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_15_load_model_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
print("Loading embedding model (this takes ~10 seconds on first run)...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = model.get_sentence_embedding_dimension()
print(f"Model loaded! Embedding dimension: {embedding_dim}")

In [ ]:
#@title 🎧 Listen: Embedding Sentences
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_16_embedding_sentences.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 Embedding a Diverse Sentence Collection

Let us embed a collection of 30+ sentences organized by topic. We deliberately choose diverse topics so we can later see how the embedding space clusters them.

In [ ]:
#@title 🎧 Listen: Embed Sentences Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_17_embed_sentences_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 30+ sentences organized by topic
sentences_by_topic = {
    'Science': [
        "The mitochondria is the powerhouse of the cell.",
        "Photosynthesis converts sunlight into chemical energy.",
        "DNA carries the genetic instructions for all living organisms.",
        "Gravity pulls objects toward the center of the Earth.",
        "Atoms are made of protons, neutrons, and electrons.",
        "The speed of light in a vacuum is 299,792,458 meters per second.",
        "Evolution occurs through natural selection over many generations.",
    ],
    'Cooking': [
        "Preheat the oven to 350 degrees Fahrenheit before baking.",
        "Dice the onions finely and saute them in olive oil.",
        "Let the dough rise for at least one hour in a warm place.",
        "Season the steak with salt and pepper before grilling.",
        "Fold the egg whites gently into the batter to keep it fluffy.",
        "Simmer the sauce on low heat for thirty minutes.",
        "Knead the bread dough until it becomes smooth and elastic.",
    ],
    'Sports': [
        "The goalkeeper made an incredible save in the final minute.",
        "She completed the marathon in under three hours.",
        "The basketball player scored a three-pointer at the buzzer.",
        "Tennis requires both physical endurance and mental toughness.",
        "The relay team broke the world record at the Olympics.",
        "A perfect golf swing requires proper hip rotation.",
        "The quarterback threw a sixty-yard touchdown pass.",
    ],
    'Finance': [
        "Diversifying your portfolio reduces overall investment risk.",
        "Compound interest grows your savings exponentially over time.",
        "The stock market experienced a sharp correction last quarter.",
        "Inflation erodes the purchasing power of money over time.",
        "A balanced budget requires matching revenue with expenditures.",
        "The central bank raised interest rates to control inflation.",
        "Index funds offer broad market exposure at low fees.",
        "Bond yields move inversely to bond prices.",
    ],
}

# Flatten into lists
all_sentences = []
all_topics = []
for topic, sents in sentences_by_topic.items():
    for s in sents:
        all_sentences.append(s)
        all_topics.append(topic)

print(f"Total sentences: {len(all_sentences)}")
for topic, sents in sentences_by_topic.items():
    print(f"  {topic}: {len(sents)} sentences")

# Embed all sentences
print(f"\nEmbedding {len(all_sentences)} sentences...")
start_time = time.time()
embeddings = model.encode(all_sentences, show_progress_bar=True)
embeddings = np.array(embeddings)
elapsed = time.time() - start_time

print(f"\nDone in {elapsed:.2f}s")
print(f"Embedding matrix shape: {embeddings.shape}")
print(f"  -> {embeddings.shape[0]} sentences, {embeddings.shape[1]} dimensions each")
print(f"  -> Each sentence is now a vector of {embeddings.shape[1]} numbers")

In [ ]:
#@title 🎧 Listen: Pairwise Similarity Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_18_pairwise_similarity_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 Pairwise Similarity Matrix

Now let us compute the cosine similarity between every pair of sentences and visualize the result as a heatmap. If our embedding model is any good, we should see clear blocks along the diagonal — sentences from the same topic should be more similar to each other than to sentences from other topics.

In [ ]:
#@title 🎧 Listen: Pairwise Similarity Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_19_pairwise_similarity_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

# Compute pairwise similarities
similarity_matrix = sklearn_cosine(embeddings)

# Sort sentences by topic for clean visualization
topic_order = ['Science', 'Cooking', 'Sports', 'Finance']
sorted_indices = []
sorted_labels = []
sorted_topics_ordered = []
for topic in topic_order:
    for i, t in enumerate(all_topics):
        if t == topic:
            sorted_indices.append(i)
            sorted_labels.append(all_sentences[i][:40] + '...')
            sorted_topics_ordered.append(topic)

sorted_indices = np.array(sorted_indices)
sorted_sim = similarity_matrix[np.ix_(sorted_indices, sorted_indices)]

# Plot the heatmap
fig, ax = plt.subplots(figsize=(14, 12))

im = ax.imshow(sorted_sim, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)

ax.set_xticks(range(len(sorted_labels)))
ax.set_xticklabels(sorted_labels, rotation=90, ha='right', fontsize=7)
ax.set_yticks(range(len(sorted_labels)))
ax.set_yticklabels(sorted_labels, fontsize=7)

# Draw topic boundary lines
topic_colors = {'Science': '#E91E63', 'Cooking': '#4CAF50',
                'Sports': '#2196F3', 'Finance': '#FF9800'}
boundary = 0
for topic in topic_order:
    count = sum(1 for t in sorted_topics_ordered if t == topic)
    ax.axhline(y=boundary - 0.5, color='white', linewidth=2)
    ax.axvline(x=boundary - 0.5, color='white', linewidth=2)
    # Label the block
    mid = boundary + count / 2 - 0.5
    ax.text(-2.5, mid, topic, fontsize=10, fontweight='bold',
            color=topic_colors[topic], va='center', ha='right')
    boundary += count

ax.axhline(y=boundary - 0.5, color='white', linewidth=2)
ax.axvline(x=boundary - 0.5, color='white', linewidth=2)

ax.set_title('Pairwise Similarity Matrix: Sentences Sorted by Topic\n'
             '(Bright blocks on diagonal = same-topic sentences are similar)',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Print average within-topic vs between-topic similarity
print("\nWithin-topic vs Between-topic Similarity:")
print("-" * 50)
for topic in topic_order:
    topic_mask = np.array([t == topic for t in sorted_topics_ordered])
    within = sorted_sim[np.ix_(topic_mask, topic_mask)]
    between = sorted_sim[np.ix_(topic_mask, ~topic_mask)]
    # Exclude self-similarity (diagonal)
    within_vals = within[np.triu_indices_from(within, k=1)]
    print(f"  {topic:10s}  within={np.mean(within_vals):.3f}  "
          f"between={np.mean(between):.3f}  "
          f"gap={np.mean(within_vals) - np.mean(between):.3f}")

In [ ]:
#@title 🎧 Listen: Pairwise Similarity Result
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_20_pairwise_similarity_result.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Look at those blocks along the diagonal. Sentences about cooking cluster together. Sentences about finance cluster together. And the between-topic similarities are much lower. The embedding model has learned to place semantically similar text nearby in vector space — even though it was never explicitly told about our topic categories. This is exactly what we want.

In [ ]:
#@title 🎧 Listen: Faiss Indices Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_21_faiss_indices_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.4 Building FAISS Indices

Now we arrive at the core engineering problem: given a query embedding, how do we find the most similar embeddings in a large collection? We will build three FAISS index types, each with a different tradeoff.

In [ ]:
#@title 🎧 Listen: Build Faiss Indices Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_22_build_faiss_indices_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import faiss

# --- Index 1: Flat (Brute Force, Exact) ---
# Compares query against every single vector. Perfect recall, but slow at scale.
d = embeddings.shape[1]  # 384 dimensions

index_flat = faiss.IndexFlatIP(d)  # Inner product (= cosine sim for normalized vectors)

# FAISS requires float32 and L2-normalized vectors for cosine similarity
embeddings_f32 = embeddings.astype('float32')
faiss.normalize_L2(embeddings_f32)

index_flat.add(embeddings_f32)
print(f"Flat Index: {index_flat.ntotal} vectors indexed (exact search)")

# --- Index 2: IVF (Inverted File, Cluster-Based) ---
# Partitions vectors into clusters. Only searches the nearest clusters.
N_CLUSTERS = 4  # Number of Voronoi cells (keep small for our 30 sentences)
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVF(quantizer, d, N_CLUSTERS, faiss.METRIC_INNER_PRODUCT)

# IVF requires training on the data to learn cluster centroids
index_ivf.train(embeddings_f32)
index_ivf.add(embeddings_f32)
index_ivf.nprobe = 2  # Search the 2 nearest clusters at query time
print(f"IVF Index:  {index_ivf.ntotal} vectors indexed ({N_CLUSTERS} clusters, nprobe=2)")

# --- Index 3: HNSW (Hierarchical Navigable Small World, Graph-Based) ---
# Builds a multi-layer graph. Traverses from coarse to fine for fast ANN search.
M = 16  # Number of neighbors per node in the graph
index_hnsw = faiss.IndexHNSWFlat(d, M)
index_hnsw.hnsw.efSearch = 32  # Search effort (higher = better recall, slower)
index_hnsw.add(embeddings_f32)
print(f"HNSW Index: {index_hnsw.ntotal} vectors indexed (M={M}, efSearch=32)")

print(f"\nAll three indices built with {d}-dimensional vectors!")

In [ ]:
#@title 🎧 Listen: Test Indices Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_23_test_indices_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Let Us Test All Three Indices

In [ ]:
#@title 🎧 Listen: Test Indices Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_24_test_indices_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Test query
test_query = "How does the human body produce energy?"
query_emb = model.encode([test_query]).astype('float32')
faiss.normalize_L2(query_emb)

K = 5  # Top-5 results

print(f"Query: \"{test_query}\"")
print(f"{'=' * 70}")

for idx_name, idx in [("Flat (Exact)", index_flat),
                       ("IVF (Cluster)", index_ivf),
                       ("HNSW (Graph)", index_hnsw)]:
    t0 = time.time()
    scores, indices = idx.search(query_emb, K)
    t1 = time.time()

    print(f"\n--- {idx_name} (search time: {(t1-t0)*1000:.2f} ms) ---")
    for rank, (score, i) in enumerate(zip(scores[0], indices[0]), 1):
        print(f"  {rank}. [{score:.4f}] [{all_topics[i]:8s}] {all_sentences[i][:65]}...")

In [ ]:
#@title 🎧 Listen: Benchmarking Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_25_benchmarking_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.5 Benchmarking: QPS vs Corpus Size

Now for the benchmark that matters in production. How do these indices scale as the corpus grows? We will generate synthetic embedding vectors at 1K, 10K, and 50K sizes and measure queries per second (QPS) and recall for each index type.

In [ ]:
#@title 🎧 Listen: Benchmarking Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_26_benchmarking_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Benchmark configuration
CORPUS_SIZES = [1_000, 10_000, 50_000]
D = 384
N_QUERIES = 100
K = 10

# Generate random query vectors
np.random.seed(42)
query_vectors = np.random.randn(N_QUERIES, D).astype('float32')
faiss.normalize_L2(query_vectors)

results = []

for n in CORPUS_SIZES:
    print(f"\n{'=' * 60}")
    print(f"Benchmarking with corpus size = {n:,}")
    print(f"{'=' * 60}")

    # Generate synthetic corpus
    np.random.seed(42)
    corpus = np.random.randn(n, D).astype('float32')
    faiss.normalize_L2(corpus)

    # --- Flat Index (ground truth) ---
    flat = faiss.IndexFlatIP(D)
    flat.add(corpus)

    t0 = time.time()
    gt_scores, gt_indices = flat.search(query_vectors, K)
    flat_time = time.time() - t0
    flat_qps = N_QUERIES / flat_time

    results.append({
        'corpus_size': n, 'index_type': 'Flat (Exact)',
        'qps': flat_qps, 'recall': 1.0, 'build_time': 0,
    })
    print(f"  Flat:  QPS = {flat_qps:>8.1f},  Recall = 1.000 (ground truth)")

    # --- IVF Index ---
    n_clusters = max(4, int(np.sqrt(n)))
    quantizer_bench = faiss.IndexFlatIP(D)
    ivf = faiss.IndexIVF(quantizer_bench, D, n_clusters, faiss.METRIC_INNER_PRODUCT)

    t_build = time.time()
    ivf.train(corpus)
    ivf.add(corpus)
    build_time_ivf = time.time() - t_build
    ivf.nprobe = max(1, n_clusters // 4)

    t0 = time.time()
    ivf_scores, ivf_indices = ivf.search(query_vectors, K)
    ivf_time = time.time() - t0
    ivf_qps = N_QUERIES / ivf_time

    # Compute recall: fraction of true top-K found by IVF
    ivf_recall = np.mean([
        len(set(ivf_indices[i]) & set(gt_indices[i])) / K
        for i in range(N_QUERIES)
    ])

    results.append({
        'corpus_size': n, 'index_type': 'IVF (Cluster)',
        'qps': ivf_qps, 'recall': ivf_recall, 'build_time': build_time_ivf,
    })
    print(f"  IVF:   QPS = {ivf_qps:>8.1f},  Recall = {ivf_recall:.3f} "
          f"(nlist={n_clusters}, nprobe={ivf.nprobe})")

    # --- HNSW Index ---
    hnsw = faiss.IndexHNSWFlat(D, 32)
    hnsw.hnsw.efSearch = 64
    hnsw.hnsw.efConstruction = 40

    t_build = time.time()
    hnsw.add(corpus)
    build_time_hnsw = time.time() - t_build

    t0 = time.time()
    hnsw_scores, hnsw_indices = hnsw.search(query_vectors, K)
    hnsw_time = time.time() - t0
    hnsw_qps = N_QUERIES / hnsw_time

    hnsw_recall = np.mean([
        len(set(hnsw_indices[i]) & set(gt_indices[i])) / K
        for i in range(N_QUERIES)
    ])

    results.append({
        'corpus_size': n, 'index_type': 'HNSW (Graph)',
        'qps': hnsw_qps, 'recall': hnsw_recall, 'build_time': build_time_hnsw,
    })
    print(f"  HNSW:  QPS = {hnsw_qps:>8.1f},  Recall = {hnsw_recall:.3f} "
          f"(M=32, efSearch=64)")

df_results = pd.DataFrame(results)
print(f"\nBenchmark complete! {len(results)} measurements collected.")

In [ ]:
#@title 🎧 Listen: Qps Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_27_qps_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization: QPS vs Corpus Size for All Three Index Types

This is the chart that tells the whole story. As the corpus grows, how does search speed change for each index type?

In [ ]:
#@title 🎧 Listen: Qps Viz Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_28_qps_viz_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

index_types = ['Flat (Exact)', 'IVF (Cluster)', 'HNSW (Graph)']
colors = {'Flat (Exact)': '#E91E63', 'IVF (Cluster)': '#2196F3', 'HNSW (Graph)': '#4CAF50'}
markers = {'Flat (Exact)': 'o', 'IVF (Cluster)': 's', 'HNSW (Graph)': '^'}

# Panel 1: QPS vs Corpus Size
ax1 = axes[0]
for idx_type in index_types:
    subset = df_results[df_results['index_type'] == idx_type]
    ax1.plot(subset['corpus_size'], subset['qps'],
             marker=markers[idx_type], color=colors[idx_type],
             linewidth=2.5, markersize=10, label=idx_type, zorder=3)

ax1.set_xlabel('Corpus Size (number of vectors)', fontsize=12)
ax1.set_ylabel('Queries Per Second (QPS)', fontsize=12)
ax1.set_title('Search Speed: QPS vs Corpus Size', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')
ax1.set_yscale('log')

# Annotate: HNSW advantage at large scale
max_size = CORPUS_SIZES[-1]
for idx_type in index_types:
    row = df_results[(df_results['index_type'] == idx_type) &
                     (df_results['corpus_size'] == max_size)].iloc[0]
    ax1.annotate(f"{row['qps']:.0f} QPS",
                 xy=(max_size, row['qps']),
                 xytext=(max_size * 1.5, row['qps']),
                 fontsize=9, fontweight='bold', color=colors[idx_type])

# Panel 2: Recall vs Corpus Size
ax2 = axes[1]
for idx_type in index_types:
    subset = df_results[df_results['index_type'] == idx_type]
    ax2.plot(subset['corpus_size'], subset['recall'],
             marker=markers[idx_type], color=colors[idx_type],
             linewidth=2.5, markersize=10, label=idx_type, zorder=3)

ax2.set_xlabel('Corpus Size (number of vectors)', fontsize=12)
ax2.set_ylabel('Recall@10', fontsize=12)
ax2.set_title('Search Quality: Recall vs Corpus Size', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')
ax2.set_ylim(0.5, 1.05)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.4, label='Perfect recall')

plt.suptitle('The Recall vs Speed Tradeoff in Vector Search',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Summary table
print("\nFull Benchmark Results:")
print(df_results.to_string(index=False))

In [ ]:
#@title 🎧 Listen: Qps Viz Result
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_29_qps_viz_result.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

The story this chart tells is clear: Flat index gives perfect recall but its QPS drops as the corpus grows. IVF and HNSW maintain high QPS at scale — at the cost of slightly imperfect recall. For most production systems, 95%+ recall at 10-100x the speed is a trade we gladly make.

In [ ]:
#@title 🎧 Listen: Todo 1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_30_todo_1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn


In [ ]:
#@title 🎧 Listen: Todo 1 Cosine Scratch
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_30_todo_1_cosine_scratch.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 1: Implement Cosine Similarity from Scratch

Implement cosine similarity using only numpy — no sklearn, no scipy, no shortcuts.

In [ ]:
def cosine_similarity_scratch(query_vec: np.ndarray, doc_vecs: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between a query vector and multiple document vectors.

    Args:
        query_vec: Shape (embedding_dim,) - a single query embedding
        doc_vecs:  Shape (num_docs, embedding_dim) - matrix of document embeddings

    Returns:
        Shape (num_docs,) - cosine similarity score for each document

    Formula: sim(q, d) = (q . d) / (||q|| * ||d||)

    Requirements:
        - Use ONLY numpy (no sklearn, no scipy)
        - Must handle query_vec as 1D and doc_vecs as 2D
        - Must handle the edge case of zero-norm vectors (avoid division by zero)
    """
    # ============ TODO ============
    # Step 1: Compute dot products between query and all docs
    #         Hint: doc_vecs @ query_vec gives shape (num_docs,)
    # Step 2: Compute the L2 norm (magnitude) of the query vector
    # Step 3: Compute the L2 norm of each document vector
    #         Hint: np.linalg.norm(doc_vecs, axis=1)
    # Step 4: Divide dot products by the product of norms
    # Step 5: Handle edge case: if any norm is zero, set similarity to 0
    # ==============================

    dot_products = ???    # YOUR CODE HERE
    query_norm = ???      # YOUR CODE HERE
    doc_norms = ???       # YOUR CODE HERE
    similarities = ???    # YOUR CODE HERE

    return similarities

In [ ]:
#@title 🎧 Listen: Todo 1 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_32_todo_1_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Verification: Run this cell to check your implementation

# Test 1: Identical vectors -> similarity 1.0
test_q = np.array([1.0, 0.0, 0.0])
test_d = np.array([[1.0, 0.0, 0.0]])
result = cosine_similarity_scratch(test_q, test_d)
assert np.allclose(result, [1.0], atol=1e-6), f"Test 1 failed: expected [1.0], got {result}"

# Test 2: Orthogonal vectors -> similarity 0.0
test_d2 = np.array([[0.0, 1.0, 0.0]])
result2 = cosine_similarity_scratch(test_q, test_d2)
assert np.allclose(result2, [0.0], atol=1e-6), f"Test 2 failed: expected [0.0], got {result2}"

# Test 3: Opposite vectors -> similarity -1.0
test_d3 = np.array([[-1.0, 0.0, 0.0]])
result3 = cosine_similarity_scratch(test_q, test_d3)
assert np.allclose(result3, [-1.0], atol=1e-6), f"Test 3 failed: expected [-1.0], got {result3}"

# Test 4: Batch of 3 documents
test_d4 = np.array([[1.0, 0.0, 0.0],
                     [0.0, 1.0, 0.0],
                     [-1.0, 0.0, 0.0]])
result4 = cosine_similarity_scratch(test_q, test_d4)
assert result4.shape == (3,), f"Test 4 shape failed: expected (3,), got {result4.shape}"
assert np.allclose(result4, [1.0, 0.0, -1.0], atol=1e-6), f"Test 4 values failed: got {result4}"

# Test 5: Worked example vectors from Section 3
q_art = np.array([1.0, 2.0, 3.0])
d_art = np.array([[2.0, 4.0, 6.0],
                   [3.0, -1.0, 0.0]])
result5 = cosine_similarity_scratch(q_art, d_art)
assert np.allclose(result5[0], 1.0, atol=1e-4), f"Test 5a failed: expected ~1.0, got {result5[0]}"
assert np.allclose(result5[1], 0.0845, atol=0.01), f"Test 5b failed: expected ~0.085, got {result5[1]}"

print("All 5 tests passed! Your cosine similarity implementation is correct.")

In [ ]:
#@title 🎧 Listen: Todo 2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_33_todo_2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Build an IVF Index with Configurable nprobe and Measure the Tradeoff

The `nprobe` parameter controls how many clusters IVF searches at query time. Higher nprobe means better recall but slower search. Your task: build an IVF index and sweep over different nprobe values to visualize the recall-vs-speed tradeoff.

In [ ]:
def ivf_nprobe_experiment(corpus_size: int = 10_000, d: int = 384, k: int = 10,
                          nprobe_values: list = None) -> pd.DataFrame:
    """
    Build an IVF index and measure recall and QPS at different nprobe settings.

    Args:
        corpus_size: Number of vectors in the corpus
        d: Embedding dimensionality
        k: Number of nearest neighbors to retrieve
        nprobe_values: List of nprobe values to test

    Returns:
        DataFrame with columns: nprobe, recall, qps

    Steps:
        1. Generate a random corpus and normalize it
        2. Build a Flat index as ground truth
        3. Build an IVF index with n_clusters = sqrt(corpus_size)
        4. For each nprobe value:
           a. Set index_ivf.nprobe = nprobe
           b. Search with 100 random queries
           c. Compute recall vs ground truth
           d. Compute QPS
        5. Return results as a DataFrame
    """
    if nprobe_values is None:
        nprobe_values = [1, 2, 4, 8, 16, 32, 64]

    n_queries = 100

    # ============ TODO ============
    # Step 1: Generate corpus and queries, normalize both
    # Step 2: Build Flat index -> get ground truth results
    # Step 3: Build IVF index (train + add)
    # Step 4: Loop over nprobe_values, measure recall and QPS for each
    # Step 5: Return pd.DataFrame with columns ['nprobe', 'recall', 'qps']
    # ==============================

    results = ???  # YOUR CODE HERE

    return pd.DataFrame(results)

In [ ]:
#@title 🎧 Listen: Todo 2 Verify
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_35_todo_2_verify.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Verification: Run this cell after implementing the function above

df_nprobe = ivf_nprobe_experiment(corpus_size=10_000, nprobe_values=[1, 2, 4, 8, 16, 32])

# Basic checks
assert isinstance(df_nprobe, pd.DataFrame), "Must return a DataFrame"
assert set(df_nprobe.columns) >= {'nprobe', 'recall', 'qps'}, f"Missing columns: {df_nprobe.columns}"
assert len(df_nprobe) == 6, f"Expected 6 rows, got {len(df_nprobe)}"
assert df_nprobe['recall'].iloc[-1] >= df_nprobe['recall'].iloc[0], "Recall should increase with nprobe"

# Visualize
fig, ax1 = plt.subplots(figsize=(10, 6))

color_recall = '#2196F3'
color_qps = '#FF5722'

ax1.plot(df_nprobe['nprobe'], df_nprobe['recall'], 'o-',
         color=color_recall, linewidth=2.5, markersize=8, label='Recall@10')
ax1.set_xlabel('nprobe (clusters searched)', fontsize=13)
ax1.set_ylabel('Recall@10', fontsize=13, color=color_recall)
ax1.tick_params(axis='y', labelcolor=color_recall)
ax1.set_ylim(0.5, 1.05)

ax2 = ax1.twinx()
ax2.plot(df_nprobe['nprobe'], df_nprobe['qps'], 's--',
         color=color_qps, linewidth=2.5, markersize=8, label='QPS')
ax2.set_ylabel('Queries Per Second', fontsize=13, color=color_qps)
ax2.tick_params(axis='y', labelcolor=color_qps)

ax1.set_title('IVF nprobe Sweep: The Recall vs Speed Tradeoff\n'
              'More clusters searched = better recall, slower speed',
              fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=11)

plt.tight_layout()
plt.show()

print("\nnprobe Experiment Results:")
print(df_nprobe.to_string(index=False))
print("\nAll checks passed! Your IVF experiment is correct.")

In [ ]:
#@title 🎧 Listen: Putting It Together
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_36_putting_it_together.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

Now let us build a complete semantic search engine. We will embed a corpus of 100 text chunks, index them with HNSW, and search with natural language queries.

In [ ]:
#@title 🎧 Listen: Knowledge Base Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_37_knowledge_base_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Build a knowledge base of 100 text chunks across diverse domains
knowledge_chunks = [
    # Machine Learning (20 chunks)
    "Neural networks learn by adjusting weights through backpropagation and gradient descent.",
    "Convolutional neural networks apply filters that slide across input images to detect features like edges and textures.",
    "Recurrent neural networks maintain a hidden state that carries information from previous time steps in a sequence.",
    "Transformers use self-attention mechanisms to process all positions in a sequence simultaneously.",
    "Batch normalization normalizes layer inputs during training to stabilize learning and allow higher learning rates.",
    "Dropout randomly deactivates a fraction of neurons during training to prevent overfitting.",
    "Transfer learning reuses a model pretrained on a large dataset and fine-tunes it on a smaller target dataset.",
    "The learning rate controls the step size during gradient descent and must be tuned carefully.",
    "Regularization techniques like L1 and L2 add penalty terms to the loss function to prevent overfitting.",
    "Generative adversarial networks consist of a generator and discriminator trained in competition.",
    "Attention mechanisms allow models to dynamically focus on the most relevant parts of the input.",
    "Residual connections add shortcut paths that help gradients flow through very deep networks.",
    "Data augmentation artificially increases training set size through transformations like rotation and flipping.",
    "Gradient clipping prevents exploding gradients by capping the maximum gradient norm during training.",
    "The Adam optimizer combines momentum and adaptive learning rates for efficient training.",
    "Embedding layers map discrete tokens to continuous vector representations that capture semantic meaning.",
    "Cross-validation splits data into multiple folds to evaluate model performance more robustly.",
    "The softmax function converts raw logits into a probability distribution over classes.",
    "Feature engineering involves creating new input features from raw data to improve model performance.",
    "Hyperparameter tuning searches for the best model configuration using methods like grid search or Bayesian optimization.",

    # Physics (15 chunks)
    "Newton's first law states that an object at rest stays at rest unless acted upon by an external force.",
    "The speed of light in vacuum is approximately 299,792,458 meters per second.",
    "Quantum entanglement links particles so that measuring one instantly affects the other regardless of distance.",
    "General relativity describes gravity as the curvature of spacetime caused by mass and energy.",
    "The Heisenberg uncertainty principle states you cannot precisely know both position and momentum simultaneously.",
    "Thermodynamics governs the flow of heat and the conversion between different forms of energy.",
    "Electromagnetic waves include radio waves, microwaves, visible light, and X-rays.",
    "The Standard Model describes three of the four fundamental forces and all known elementary particles.",
    "Nuclear fusion combines light atomic nuclei to release enormous amounts of energy, as occurs in stars.",
    "Superconductors conduct electricity with zero resistance below a critical temperature.",
    "Dark matter makes up about 27 percent of the universe but does not interact with light.",
    "The photoelectric effect demonstrated that light consists of discrete packets of energy called photons.",
    "Conservation of energy states that energy cannot be created or destroyed, only transformed.",
    "Black holes are regions of spacetime where gravity is so strong that nothing can escape.",
    "Particle accelerators collide subatomic particles at near-light speeds to study fundamental physics.",

    # History (15 chunks)
    "The printing press, invented by Gutenberg around 1440, revolutionized the spread of knowledge in Europe.",
    "The French Revolution of 1789 overthrew the monarchy and established principles of liberty and equality.",
    "The Industrial Revolution transformed manufacturing from hand production to machine-based production.",
    "World War II lasted from 1939 to 1945 and involved most of the world's nations.",
    "The fall of the Berlin Wall in 1989 symbolized the end of the Cold War.",
    "Ancient Egypt built the pyramids as monumental tombs for their pharaohs.",
    "The Renaissance was a cultural movement that began in Italy and spread throughout Europe in the 14th century.",
    "The moon landing in 1969 marked the first time humans set foot on another celestial body.",
    "The Roman Empire at its peak controlled territory spanning three continents.",
    "The Silk Road was an ancient network of trade routes connecting East Asia to the Mediterranean.",
    "The invention of the steam engine powered the Industrial Revolution and transformed transportation.",
    "The Declaration of Independence was adopted on July 4, 1776, establishing the United States.",
    "The Great Wall of China was built over centuries to protect against invasions from the north.",
    "The discovery of penicillin by Alexander Fleming in 1928 revolutionized medicine.",
    "The Age of Exploration in the 15th century led European sailors to discover new continents.",

    # Biology (15 chunks)
    "DNA is a double helix molecule that carries genetic instructions for development and function.",
    "Mitochondria are organelles that generate most of the cell's supply of adenosine triphosphate.",
    "Photosynthesis converts carbon dioxide and water into glucose and oxygen using sunlight.",
    "Evolution through natural selection favors organisms best adapted to their environment.",
    "The human genome contains approximately 3 billion base pairs of DNA.",
    "Antibodies are proteins produced by the immune system to identify and neutralize pathogens.",
    "Neurons communicate through electrical impulses and chemical neurotransmitters across synapses.",
    "Cell division through mitosis produces two genetically identical daughter cells.",
    "Enzymes are biological catalysts that speed up chemical reactions in living organisms.",
    "The circulatory system transports oxygen, nutrients, and waste products throughout the body.",
    "Genetic mutations can be harmful, neutral, or beneficial depending on their effect on fitness.",
    "Stem cells have the ability to develop into many different types of specialized cells.",
    "Photoreceptors in the retina convert light into electrical signals that the brain interprets as vision.",
    "The human brain contains approximately 86 billion neurons connected by trillions of synapses.",
    "Ecosystems maintain balance through food chains and nutrient cycles.",

    # Cooking (15 chunks)
    "Caramelization occurs when sugar is heated above 320 degrees Fahrenheit, creating complex flavors.",
    "The Maillard reaction between amino acids and reducing sugars gives browned food its distinctive flavor.",
    "Emulsification combines two immiscible liquids, like oil and vinegar, into a stable mixture.",
    "Fermentation uses microorganisms to convert sugars into alcohol or acids, as in bread and yogurt.",
    "Blanching briefly boils vegetables then plunges them into ice water to preserve color and texture.",
    "Braising slowly cooks tough cuts of meat in liquid until they become tender and flavorful.",
    "A roux is a mixture of equal parts fat and flour used to thicken sauces and soups.",
    "Mise en place means preparing and organizing all ingredients before starting to cook.",
    "Deglazing adds liquid to a hot pan to dissolve caramelized food residue into a flavorful sauce.",
    "Proofing yeast in warm water with sugar activates it before adding to dough.",
    "Sous vide cooking seals food in a vacuum bag and cooks it in precisely controlled water temperatures.",
    "The five basic French mother sauces are bechamel, veloute, espagnole, hollandaise, and tomato.",
    "Tempering chocolate involves carefully heating and cooling to achieve a smooth glossy finish.",
    "Gluten develops when flour proteins are hydrated and kneaded, giving bread its chewy structure.",
    "Acid from citrus or vinegar can chemically cook proteins in ceviche without heat.",

    # Finance (20 chunks)
    "Compound interest calculates returns on both the principal and previously accumulated interest.",
    "Diversification spreads investments across different asset classes to reduce portfolio risk.",
    "The price-to-earnings ratio compares a company's stock price to its earnings per share.",
    "Inflation reduces the purchasing power of money over time, increasing the cost of goods.",
    "A bear market is defined as a decline of 20 percent or more from recent highs.",
    "Dollar-cost averaging invests a fixed amount at regular intervals regardless of market conditions.",
    "Bonds are debt instruments where investors lend money to an entity in exchange for periodic interest.",
    "The Federal Reserve sets monetary policy by adjusting interest rates and the money supply.",
    "An initial public offering allows a private company to raise capital by selling shares to the public.",
    "Liquidity refers to how quickly an asset can be converted to cash without significant loss of value.",
    "A recession is typically defined as two consecutive quarters of declining gross domestic product.",
    "Market capitalization equals the stock price multiplied by the total number of outstanding shares.",
    "Hedge funds use alternative strategies like short selling and leverage to generate returns.",
    "A yield curve plots bond yields across different maturities and can signal economic conditions.",
    "Working capital is the difference between a company's current assets and current liabilities.",
    "Passive investing through index funds aims to match market returns rather than beat them.",
    "Credit risk is the probability that a borrower will default on their financial obligations.",
    "Venture capital provides funding to early-stage startups in exchange for equity ownership.",
    "The efficient market hypothesis suggests that asset prices reflect all available information.",
    "Cryptocurrency operates on decentralized blockchain technology without central bank control.",
]

print(f"Knowledge base: {len(knowledge_chunks)} chunks")
print(f"  Machine Learning: 20 chunks")
print(f"  Physics: 15 chunks")
print(f"  History: 15 chunks")
print(f"  Biology: 15 chunks")
print(f"  Cooking: 15 chunks")
print(f"  Finance: 20 chunks")

In [ ]:
#@title 🎧 Listen: Embed Index Kb Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_38_embed_index_kb_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Embed the knowledge base and build an HNSW index
print("Embedding 100 knowledge chunks...")
kb_embeddings = model.encode(knowledge_chunks, show_progress_bar=True)
kb_embeddings = np.array(kb_embeddings, dtype='float32')
faiss.normalize_L2(kb_embeddings)

# Build HNSW index for fast search
D = kb_embeddings.shape[1]
search_index = faiss.IndexHNSWFlat(D, 32)
search_index.hnsw.efSearch = 64
search_index.add(kb_embeddings)

print(f"\nHNSW index built: {search_index.ntotal} vectors, {D} dimensions")

In [ ]:
#@title 🎧 Listen: Semantic Search Func Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_39_semantic_search_func_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def semantic_search(query: str, top_k: int = 5) -> List[Tuple[str, float]]:
    """
    Search the knowledge base using semantic similarity.

    Args:
        query: Natural language search query
        top_k: Number of results to return

    Returns:
        List of (chunk_text, similarity_score) tuples
    """
    # Embed the query
    query_emb = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_emb)

    # Search the HNSW index
    scores, indices = search_index.search(query_emb, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append((knowledge_chunks[idx], float(score)))

    return results


# Test with a diverse set of queries
test_queries = [
    "How do machines learn from data?",
    "What happens inside a star?",
    "How did ancient civilizations trade with each other?",
    "What makes bread rise?",
    "How can I protect my investments during a market downturn?",
]

print("=" * 70)
print("SEMANTIC SEARCH ENGINE — Top-5 Results")
print("=" * 70)

for query in test_queries:
    results = semantic_search(query, top_k=5)
    print(f"\nQuery: \"{query}\"")
    print("-" * 60)
    for rank, (chunk, score) in enumerate(results, 1):
        print(f"  {rank}. [{score:.4f}] {chunk[:75]}...")
    print()

In [ ]:
#@title 🎧 Listen: Semantic Search Result
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_40_semantic_search_result.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Notice how the search engine finds semantically relevant results even when the query uses completely different words than the chunks. "How do machines learn from data?" retrieves documents about neural networks, gradient descent, and backpropagation — none of which literally contain the word "machines" or "learn from data." This is the power of embedding-based search.

In [ ]:
#@title 🎧 Listen: Embedding Quality Analysis
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_41_embedding_quality_analysis.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Embedding Quality Analysis

Let us explore what the embedding space actually looks like by projecting our high-dimensional embeddings down to 2D.

### t-SNE Projection Colored by Topic

In [ ]:
#@title 🎧 Listen: Tsne Projection Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_42_tsne_projection_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
from sklearn.manifold import TSNE

# Assign topic labels to all 100 chunks
chunk_topics = (
    ['ML'] * 20 +
    ['Physics'] * 15 +
    ['History'] * 15 +
    ['Biology'] * 15 +
    ['Cooking'] * 15 +
    ['Finance'] * 20
)

# Project from 384 dimensions to 2 dimensions using t-SNE
print("Running t-SNE projection (this may take a few seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=15, n_iter=1000)
embeddings_2d = tsne.fit_transform(kb_embeddings)
print("Done!")

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

topic_colors_map = {
    'ML': '#E91E63',
    'Physics': '#2196F3',
    'History': '#FF9800',
    'Biology': '#4CAF50',
    'Cooking': '#9C27B0',
    'Finance': '#00BCD4',
}

for topic, color in topic_colors_map.items():
    mask = [t == topic for t in chunk_topics]
    points = embeddings_2d[mask]
    ax.scatter(points[:, 0], points[:, 1], c=color, label=topic,
               s=100, alpha=0.7, edgecolors='white', linewidth=0.5, zorder=3)

    # Add centroid label
    centroid = points.mean(axis=0)
    ax.annotate(topic, xy=centroid, fontsize=13, fontweight='bold',
                color=color, ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=color, alpha=0.8))

ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('100 Knowledge Chunks Projected to 2D\n'
             'Chunks about the same topic cluster together',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Ambiguous Queries Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_43_ambiguous_queries_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Nearest Neighbors for Ambiguous Queries

Let us see what happens when we search for queries that could belong to multiple topics. These edge cases reveal how the embedding model handles ambiguity.

In [ ]:
#@title 🎧 Listen: Ambiguous Queries Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_44_ambiguous_queries_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
ambiguous_queries = [
    "energy",                          # Physics or Biology or Cooking?
    "networks",                        # ML or History (trade networks)?
    "growth",                          # Biology or Finance?
    "models",                          # ML or Fashion?
    "revolution",                      # History or ML (AI revolution)?
    "How does temperature affect reactions?",  # Chemistry/Physics or Cooking?
]

print("=" * 70)
print("AMBIGUOUS QUERY ANALYSIS")
print("What topics do the nearest neighbors come from?")
print("=" * 70)

for query in ambiguous_queries:
    results = semantic_search(query, top_k=5)
    # Count which topics appear in top-5
    result_topics = []
    for chunk, score in results:
        idx = knowledge_chunks.index(chunk)
        result_topics.append(chunk_topics[idx])

    topic_counts = {}
    for t in result_topics:
        topic_counts[t] = topic_counts.get(t, 0) + 1

    print(f"\nQuery: \"{query}\"")
    print(f"  Top-5 topic distribution: {dict(topic_counts)}")
    for rank, (chunk, score) in enumerate(results[:3], 1):
        idx = knowledge_chunks.index(chunk)
        print(f"  {rank}. [{score:.4f}] [{chunk_topics[idx]:8s}] {chunk[:60]}...")

In [ ]:
#@title 🎧 Listen: Tsne Queries Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_45_tsne_queries_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization: 2D Projection with Query Points

Let us project our ambiguous queries into the same 2D space and see where they land relative to the topic clusters.

In [ ]:
#@title 🎧 Listen: Tsne Queries Viz Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_46_tsne_queries_viz_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Embed the ambiguous queries
query_embs = model.encode(ambiguous_queries).astype('float32')

# Project queries into the same t-SNE space
# Note: we refit t-SNE with queries included for consistent projection
all_vecs = np.vstack([kb_embeddings, query_embs])
print("Running t-SNE with queries included...")
tsne_all = TSNE(n_components=2, random_state=42, perplexity=15, n_iter=1000)
all_2d = tsne_all.fit_transform(all_vecs)

chunk_2d = all_2d[:len(kb_embeddings)]
query_2d = all_2d[len(kb_embeddings):]
print("Done!")

fig, ax = plt.subplots(figsize=(14, 10))

# Plot chunk points
for topic, color in topic_colors_map.items():
    mask = [t == topic for t in chunk_topics]
    points = chunk_2d[mask]
    ax.scatter(points[:, 0], points[:, 1], c=color, label=topic,
               s=80, alpha=0.5, edgecolors='white', linewidth=0.5)

# Plot query points with prominent markers
for i, (query, point) in enumerate(zip(ambiguous_queries, query_2d)):
    ax.scatter(point[0], point[1], c='black', s=200, marker='*',
               zorder=5, edgecolors='yellow', linewidth=1.5)
    label = query if len(query) <= 20 else query[:20] + '...'
    ax.annotate(label, xy=point, xytext=(10, 10),
                textcoords='offset points', fontsize=9, fontweight='bold',
                color='black', bbox=dict(boxstyle='round,pad=0.2',
                                          facecolor='yellow', alpha=0.8))

ax.set_xlabel('t-SNE Dimension 1', fontsize=12)
ax.set_ylabel('t-SNE Dimension 2', fontsize=12)
ax.set_title('Ambiguous Queries in Embedding Space\n'
             '(Stars = query points, circles = knowledge chunks)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
#@title 🎧 Listen: Final Output Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_47_final_output_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Final Output

Let us run a final interactive-style search demo. We will process five diverse queries and present the results in a polished format with similarity scores.

In [ ]:
#@title 🎧 Listen: Final Output Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_48_final_output_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
demo_queries = [
    "What technique prevents neural networks from memorizing training data?",
    "How do stars produce energy through nuclear reactions?",
    "What historical event ended the Cold War between East and West?",
    "How does the human immune system fight off infections?",
    "What is the best strategy for long-term wealth building?",
]

print("=" * 75)
print("  SEMANTIC SEARCH ENGINE — FINAL DEMO")
print("  100 chunks | 384-dimensional embeddings | HNSW index")
print("=" * 75)

for q_idx, query in enumerate(demo_queries, 1):
    results = semantic_search(query, top_k=5)

    print(f"\n{'─' * 75}")
    print(f"  Query {q_idx}: \"{query}\"")
    print(f"{'─' * 75}")

    for rank, (chunk, score) in enumerate(results, 1):
        idx = knowledge_chunks.index(chunk)
        topic = chunk_topics[idx]

        # Highlight: similarity bar
        bar_len = int(score * 40)
        bar = '█' * bar_len + '░' * (40 - bar_len)

        print(f"\n  Rank {rank}  |  Similarity: {score:.4f}  [{bar}]")
        print(f"  Topic:  {topic}")
        print(f"  Text:   {chunk}")

print(f"\n{'=' * 75}")
print(f"  Search complete. 5 queries processed over 100 knowledge chunks.")
print(f"{'=' * 75}")

In [ ]:
#@title 🎧 Listen: Revisit Automobile Problem Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_49_revisit_automobile_problem_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Let Us Revisit the "Automobile" Problem

Remember where we started? Keyword search could not connect "automobile" to "car." Let us see how our embedding-based search handles this and similar cases.

In [ ]:
#@title 🎧 Listen: Revisit Automobile Problem Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_50_revisit_automobile_problem_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# The original problem + more synonym tests
synonym_tests = [
    ("automobile maintenance", "car"),
    ("canine behavior", "dog"),
    ("monetary policy", "money"),
    ("celestial bodies", "stars"),
    ("culinary techniques", "cooking"),
]

print("Synonym Resolution Test")
print("=" * 65)
print("Can embedding search bridge the vocabulary gap?")
print()

for query, expected_keyword in synonym_tests:
    results = semantic_search(query, top_k=3)
    # Check if any top-3 result contains the expected keyword
    found = any(expected_keyword.lower() in chunk.lower() for chunk, _ in results)

    status = "PASS" if found else "MISS"
    print(f"  [{status}] Query: \"{query}\"")
    print(f"        Looking for chunks about '{expected_keyword}'")
    for rank, (chunk, score) in enumerate(results[:2], 1):
        contains = f" <-- contains '{expected_keyword}'" if expected_keyword.lower() in chunk.lower() else ""
        print(f"        {rank}. [{score:.4f}] {chunk[:60]}...{contains}")
    print()

print("Embedding search understands that 'automobile' and 'car' are the same")
print("concept — something keyword search fundamentally cannot do.")

In [ ]:
#@title 🎧 Listen: Reflection And Next Steps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_51_reflection_and_next_steps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What Did We Build?

We built a complete embedding and vector search pipeline from the ground up:

1. **Cosine similarity** — implemented from scratch, verified against worked examples
2. **Sentence embeddings** — using a pretrained model that maps text to 384-dimensional vectors
3. **Pairwise similarity analysis** — showing that same-topic sentences cluster automatically
4. **Three FAISS index types** — Flat (exact), IVF (cluster-based), HNSW (graph-based)
5. **Benchmarking** — measuring the recall vs QPS tradeoff at scale
6. **A working semantic search engine** — 100 chunks, natural language queries, ranked results
7. **Embedding space analysis** — t-SNE projections showing how topics cluster

### Reflection Questions

Take a moment to think about these:

1. **Why does cosine similarity work better than Euclidean distance for comparing embeddings?** Think about what happens when one document is much longer than another — how does this affect the magnitude of its embedding vector?

2. **The HNSW index achieved 95%+ recall while being much faster than brute force. What is it giving up?** What kinds of queries might return different results from HNSW vs exact search?

3. **Our embedding model was trained on general text. What would happen if we used it for a highly specialized domain like medical imaging reports or legal contracts?** Would the similarity scores still be meaningful? What could we do about it?

### Optional Challenges

If you want to go deeper, try these:

1. **Hybrid search**: Implement a search that combines keyword matching (count shared words) with embedding similarity. Use a weighted sum: `final_score = alpha * embedding_sim + (1 - alpha) * keyword_score`. Try different values of alpha and compare results.

2. **Custom embedding fine-tuning**: Take 20 sentence pairs that you label as "similar" and 20 pairs you label as "dissimilar." Compute the model's similarity scores for each pair. How well does the pretrained model agree with your labels? Where does it fail?

3. **Quantization benchmark**: FAISS supports product quantization (PQ) which compresses vectors to use less memory. Build an `IndexIVFPQ` and measure how much recall drops compared to `IndexIVFFlat`. At what compression ratio does recall become unacceptable?

In [ ]:
#@title 🎧 Listen: Key Takeaways
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/03_52_key_takeaways.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("Notebook complete!")
print()
print("Key takeaways:")
print("  1. Embeddings map text to vectors where similar meanings are nearby")
print("  2. Cosine similarity measures direction, not magnitude — exactly what we need")
print("  3. FAISS gives us three index types with different speed/recall tradeoffs")
print("  4. Brute force is fine for small corpora; HNSW shines at scale")
print("  5. Embedding search solves the 'automobile vs car' problem that keyword search cannot")

**Estimated time spent: ~50 minutes**

---

*Part 3 of the Vizuara series on RAG Systems*